# The molecule catalogue {#sec-molecules}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/a03-molecules.ipynb)

The chapters build their models from **molecules**: small, reusable models of one recurring structure, such as a stock that drains in proportion to itself, a pipeline of orders, or a judgement anchored on a reference value. This appendix lists all 73 that ship with the `loopbricks` package, grouped into families, with each one's equations, settings, presets and the chapter where it is explained.

**Where they come from.** 71 of the molecules are our own implementations of structures described by Jim Hines in *Molecules of Structure* [@hines_molecules]; the equations, defaults and wording are ours, and Hines is credited on every entry. The other two, Tragedy of the Commons and Shifting the Burden, are our own models of two well-known system archetypes (@sec-patterns).

**How to read an entry.** Each entry gives the molecule's name and its key, the short name you pass to `lb.molecule()`; the chapter that explains it (its **home chapter**; other chapters use it freely afterwards); a one-paragraph description; its equations, printed by `equations()` as in @sec-package; a table of the constants you can set in `run()`, with their default values, the slider range the LoopBricks app offers, and units; and its presets, named settings you can run with `run(preset=...)`.

**This appendix is generated.** Every entry below is written by the code in this notebook from the model files in the package, so it always matches what `lb.molecule()` loads. Each family also shows the default run of its molecules, so you can see at a glance what each one does. In the web book the code is hidden; open the notebook in Colab to see it or to change it.

In [ ]:
# Setup: run this cell first. In Colab it installs loopbricks (ten seconds).
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/"
                           "loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import warnings
import numpy as np
import matplotlib.pyplot as plt
from cycler import cycler

# Okabe-Ito colours with a distinct line style each, readable in greyscale
plt.rcParams["axes.prop_cycle"] = (
    cycler(color=["#000000", "#E69F00", "#56B4E9", "#009E73",
                  "#0072B2", "#D55E00", "#CC79A7"])
    + cycler(linestyle=["-", "--", ":", "-.", (0, (5, 1)),
                        (0, (3, 1, 1, 1)), (0, (1, 1))]))

# the families, in the order the LoopBricks app's library shows them
FAMILIES = [
    ('Accumulation primitives', ['bathtub', 'cascade', 'conversion', 'split', 'broken']),
    ('Delays and decays', ['goToZero', 'decay', 'residence', 'material', 'aging', 'smooth', 'smoothHigher']),
    ('Goal-seeking and control', ['closegap', 'stockAdjust', 'stockmgmt', 'lowVisPipeline', 'workforce']),
    ('Expectations and valuation', ['trend', 'extrapolation', 'weightedAvg', 'presentValue']),
    ('Coflows', ['coflow', 'coflowExperience', 'agingPDY', 'cascadedCoflow']),
    ('Growth and limits', ['growth', 'logistic', 'diffusion']),
    ('Constraints and nonlinearity', ['ceiling', 'floor', 'capacityUtil', 'effectFunction']),
    ('Protected levels and fulfilment', ['protLevel', 'protFlow', 'backlogFlow', 'backlogLevel']),
    ('Anchoring and pricing', ['univariateAnchor', 'multivariateAnchor', 'seaAnchor', 'protectedSeaAnchor', 'seaAnchorPricing', 'protSeaAnchorPricing', 'smoothPricing']),
    ('Allocation and competition', ['propSplit', 'weightedSplit', 'multiSplit', 'nonlinearSplit', 'marketshare']),
    ('Resources and actions', ['actionFromResource', 'financialFlow', 'resourcesFromAction', 'workforceFromBudget', 'abilityFromAction']),
    ('Productivity and projects', ['prodFatigue', 'productivity', 'effectFatigue', 'quality', 'doingWork', 'producing', 'reducingBacklog', 'estProductivity', 'desiredWorkforce', 'overtime', 'protByPDY', 'buildingInventory', 'doingWorkCascade', 'cascadeProtByPDY', 'reworkCycle', 'estRemaining', 'estCompletion', 'schedulePressure']),
    ('System archetypes', ['tragedyCommons', 'shiftingBurden']),
]
# home chapter of each molecule: where the book explains it
HOME = {
    **dict.fromkeys([
        'bathtub', 'broken', 'conversion', 'split'], 1),
    **dict.fromkeys([
        'ceiling', 'closegap', 'decay', 'diffusion', 'floor', 'goToZero', 'growth',
        'logistic', 'marketshare', 'residence'], 3),
    **dict.fromkeys([
        'aging', 'agingPDY', 'cascade', 'cascadedCoflow', 'coflow',
        'coflowExperience', 'extrapolation', 'material', 'smooth', 'smoothHigher',
        'trend', 'weightedAvg'], 4),
    **dict.fromkeys([
        'abilityFromAction', 'actionFromResource', 'backlogFlow', 'backlogLevel',
        'capacityUtil', 'doingWork', 'effectFunction', 'estProductivity',
        'financialFlow', 'multiSplit', 'multivariateAnchor', 'nonlinearSplit',
        'presentValue', 'producing', 'productivity', 'propSplit', 'protFlow',
        'protLevel', 'protectedSeaAnchor', 'resourcesFromAction', 'seaAnchor',
        'stockAdjust', 'univariateAnchor', 'weightedSplit', 'workforceFromBudget'], 5),
    **dict.fromkeys([
        'buildingInventory', 'desiredWorkforce', 'lowVisPipeline',
        'protSeaAnchorPricing', 'seaAnchorPricing', 'smoothPricing', 'stockmgmt',
        'workforce'], 6),
    **dict.fromkeys([
        'effectFatigue', 'overtime', 'prodFatigue', 'quality', 'reworkCycle',
        'schedulePressure', 'shiftingBurden', 'tragedyCommons'], 7),
    **dict.fromkeys([
        'cascadeProtByPDY', 'doingWorkCascade', 'protByPDY', 'reducingBacklog'], 9),
    **dict.fromkeys([
        'estCompletion', 'estRemaining'], 11),
}
CHAPTER = {1: "sec-push-back", 2: "sec-story-to-diagram", 3: "sec-loops", 4: "sec-delays",
           5: "sec-decisions", 6: "sec-swings", 7: "sec-patterns", 9: "sec-testing",
           11: "sec-model-to-decision"}
OWN = {"tragedyCommons", "shiftingBurden"}   # our own archetype models, not Hines's
assert sorted(k for _, keys in FAMILIES for k in keys) == lb.molecules() == sorted(HOME)
print(len(lb.molecules()), "molecules in", len(FAMILIES), "families")

In [ ]:
def title(m):
    """The molecule's title in the book's style: 'and' rather than '&'."""
    return m.spec.get("title", m.name).replace(" & ", " and ")


def entry(key):
    """One catalogue entry, as Markdown, built from the model file."""
    m = lb.molecule(key)
    names = {e["id"]: lb.name_key(e["name"]) for e in m.vars}   # preset ids to Python names
    units = {lb.name_key(e["name"]): e.get("units", "") for e in m.vars}
    credit = ("Our own model of a system archetype."
              if key in OWN else "After Hines, *Molecules of Structure* [@hines_molecules].")
    out = [f"### {title(m)} {{#sec-mol-{key}}}", "",
           f"`lb.molecule(\"{key}\")`. Home chapter: @{CHAPTER[HOME[key]]}. "
           f"Time in {m.time_unit}s; default run {m.T:g} with steps of {m.dt:g}. {credit}", "",
           m.spec.get("description", ""), "", "```text", m.equations(), "```", ""]
    if m.params:
        out += ["| Constant | Default | App range | Units |", "|---|---|---|---|"]
        for name, value in m.params.items():
            r = m.ranges.get(name)
            span = f"{r['min']:g} to {r['max']:g}" if r else ""
            out.append(f"| `{name}` | {value:g} | {span} | {units.get(name, '')} |")
        out.append("")
    if m.presets:
        out.append("Presets (settings that differ from the defaults):")
        out.append("")
        for p in m.presets:
            changed = {names.get(k, k): v for k, v in p["params"].items()
                       if v != m.params.get(names.get(k, k))}    # only what differs from the defaults
            how = ", ".join(f"`{k}` = {v:g}" for k, v in changed.items()) or "the defaults"
            out.append(f"- **{p['label']}**: {how}")
        out.append("")
    return "\n".join(out)


def show_family(i):
    """Default runs of one family, a panel per molecule."""
    name, keys = FAMILIES[i]
    cols = 3
    rows = -(-len(keys) // cols)                  # rounds up: enough rows for every panel
    fig, axes = plt.subplots(rows, cols, figsize=(10, 2.4 * rows), squeeze=False)
    for ax, key in zip(axes.flat, keys):
        m = lb.molecule(key)
        with warnings.catch_warnings():           # a floor at zero is part of some designs
            warnings.simplefilter("ignore")
            run = m.run(include="stocks")
        run.plot(ax=ax, legend=len(run.columns) <= 4, lw=1.4)
        ax.set_title(title(m), fontsize=9)
        ax.set_xlabel(f"time ({m.time_unit}s)", fontsize=8)
        ax.tick_params(labelsize=7)
        if ax.get_legend():
            ax.legend(fontsize=6, loc="best")
    for ax in axes.flat[len(keys):]:
        ax.axis("off")                            # empty panels at the end of the grid
    plt.tight_layout()
    plt.show()


def show_entries(i):
    """The catalogue entries of one family, as Markdown."""
    print("\n\n".join(entry(k) for k in FAMILIES[i][1]))

Here is every molecule at a glance, with its family and home chapter.

In [ ]:
#| output: asis
#| echo: false
print("| Molecule | Key | Family | Home chapter |")
print("|---|---|---|---|")
for fam, keys in FAMILIES:
    for key in keys:
        m = lb.molecule(key)
        print(f"| [{title(m)}](#sec-mol-{key}) | `{key}` | {fam} | @{CHAPTER[HOME[key]]} |")

## 1. Accumulation primitives

Each panel shows one molecule's stocks over its default run; the entries follow.

In [ ]:
#| echo: false
#| fig-alt: "Default runs of the 5 molecules in the accumulation primitives family, one small panel each, showing their stocks over time."
show_family(0)

In [ ]:
#| output: asis
#| echo: false
show_entries(0)

## 2. Delays and decays

Each panel shows one molecule's stocks over its default run; the entries follow.

In [ ]:
#| echo: false
#| fig-alt: "Default runs of the 7 molecules in the delays and decays family, one small panel each, showing their stocks over time."
show_family(1)

In [ ]:
#| output: asis
#| echo: false
show_entries(1)

## 3. Goal-seeking and control

Each panel shows one molecule's stocks over its default run; the entries follow.

In [ ]:
#| echo: false
#| fig-alt: "Default runs of the 5 molecules in the goal-seeking and control family, one small panel each, showing their stocks over time."
show_family(2)

In [ ]:
#| output: asis
#| echo: false
show_entries(2)

## 4. Expectations and valuation

Each panel shows one molecule's stocks over its default run; the entries follow.

In [ ]:
#| echo: false
#| fig-alt: "Default runs of the 4 molecules in the expectations and valuation family, one small panel each, showing their stocks over time."
show_family(3)

In [ ]:
#| output: asis
#| echo: false
show_entries(3)

## 5. Coflows

Each panel shows one molecule's stocks over its default run; the entries follow.

In [ ]:
#| echo: false
#| fig-alt: "Default runs of the 4 molecules in the coflows family, one small panel each, showing their stocks over time."
show_family(4)

In [ ]:
#| output: asis
#| echo: false
show_entries(4)

## 6. Growth and limits

Each panel shows one molecule's stocks over its default run; the entries follow.

In [ ]:
#| echo: false
#| fig-alt: "Default runs of the 3 molecules in the growth and limits family, one small panel each, showing their stocks over time."
show_family(5)

In [ ]:
#| output: asis
#| echo: false
show_entries(5)

## 7. Constraints and nonlinearity

Each panel shows one molecule's stocks over its default run; the entries follow.

In [ ]:
#| echo: false
#| fig-alt: "Default runs of the 4 molecules in the constraints and nonlinearity family, one small panel each, showing their stocks over time."
show_family(6)

In [ ]:
#| output: asis
#| echo: false
show_entries(6)

## 8. Protected levels and fulfilment

Each panel shows one molecule's stocks over its default run; the entries follow.

In [ ]:
#| echo: false
#| fig-alt: "Default runs of the 4 molecules in the protected levels and fulfilment family, one small panel each, showing their stocks over time."
show_family(7)

In [ ]:
#| output: asis
#| echo: false
show_entries(7)

## 9. Anchoring and pricing

Each panel shows one molecule's stocks over its default run; the entries follow.

In [ ]:
#| echo: false
#| fig-alt: "Default runs of the 7 molecules in the anchoring and pricing family, one small panel each, showing their stocks over time."
show_family(8)

In [ ]:
#| output: asis
#| echo: false
show_entries(8)

## 10. Allocation and competition

Each panel shows one molecule's stocks over its default run; the entries follow.

In [ ]:
#| echo: false
#| fig-alt: "Default runs of the 5 molecules in the allocation and competition family, one small panel each, showing their stocks over time."
show_family(9)

In [ ]:
#| output: asis
#| echo: false
show_entries(9)

## 11. Resources and actions

Each panel shows one molecule's stocks over its default run; the entries follow.

In [ ]:
#| echo: false
#| fig-alt: "Default runs of the 5 molecules in the resources and actions family, one small panel each, showing their stocks over time."
show_family(10)

In [ ]:
#| output: asis
#| echo: false
show_entries(10)

## 12. Productivity and projects

Each panel shows one molecule's stocks over its default run; the entries follow.

In [ ]:
#| echo: false
#| fig-alt: "Default runs of the 18 molecules in the productivity and projects family, one small panel each, showing their stocks over time."
show_family(11)

In [ ]:
#| output: asis
#| echo: false
show_entries(11)

## 13. System archetypes

Each panel shows one molecule's stocks over its default run; the entries follow.

In [ ]:
#| echo: false
#| fig-alt: "Default runs of the 2 molecules in the system archetypes family, one small panel each, showing their stocks over time."
show_family(12)

In [ ]:
#| output: asis
#| echo: false
show_entries(12)